<a href="https://colab.research.google.com/github/behraj/workshops/blob/main/H_pylori_Detection_Baselines.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# H. pylori Detection Baselines

## Two Frozen Feature Extractor Models:
1. **Baseline 1**: Frozen ResNet-50 (ImageNet pre-trained) + Linear Classifier
2. **Baseline 2**: Frozen DINO ViT (Self-supervised) + Linear Classifier

### Features:
- Input: 224×224 RGB endoscopy images
- Binary classification: H. pylori +/−
- Metrics: AUC-ROC, Accuracy, Sensitivity, Specificity, ECE
- Dataset: HyperKvasir (or custom dataset)

---

## 1. Setup & Installation

First, let's check if we have GPU access and install required packages.

In [ ]:
# Check GPU availability
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"CUDA version: {torch.version.cuda}")
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB")
else:
    print("⚠️ No GPU found - training will be slower on CPU")

PyTorch version: 2.9.0+cu128
CUDA available: True
CUDA version: 12.8
GPU: Tesla T4
GPU Memory: 15.64 GB


In [ ]:
# Install required packages
!pip install -q torch torchvision
!pip install -q scikit-learn matplotlib Pillow tqdm

print("✓ All packages installed successfully!")

✓ All packages installed successfully!


### MedMNIST

In [ ]:
!pip install medmnist

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 115.9/115.9 kB 8.9 MB/s eta 0:00:00


## 2. Dataset Preparation

### Option A: Upload Your Own Dataset
If you have your own labeled dataset, upload it with this structure:
```
dataset.zip
  ├── train/
  │   ├── positive/  # H. pylori positive images
  │   └── negative/  # H. pylori negative images
  ├── val/
  │   ├── positive/
  │   └── negative/
  └── test/
      ├── positive/
      └── negative/
```

### Option B: Use HyperKvasir
Download from: https://datasets.simula.no/hyper-kvasir/

**Note**: HyperKvasir doesn't have explicit H. pylori labels. You'll need to define which classes represent positive/negative cases.

In [ ]:
# Create directories
import os
os.makedirs('/content/data', exist_ok=True)
os.makedirs('/content/results', exist_ok=True)

print("Upload your dataset as a ZIP file using the file browser on the left")
print("Or mount Google Drive to access your data:")
print("")
print("from google.colab import drive")
print("drive.mount('/content/drive')")

Upload your dataset as a ZIP file using the file browser on the left
Or mount Google Drive to access your data:

from google.colab import drive
drive.mount('/content/drive')


### Kaggle dataset

In [ ]:
import kagglehub

# Download latest version

path = kagglehub.dataset_download("paultimothymooney/chest-xray-pneumonia")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'chest-xray-pneumonia' dataset.
Path to dataset files: /kaggle/input/chest-xray-pneumonia


In [ ]:
# Optional: Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

print("✓ Google Drive mounted at /content/drive")
print("You can now access files from your Drive")

In [ ]:
# Extract uploaded dataset (if using ZIP)
import zipfile

# Modify this path to point to your ZIP file
# zip_path = '/content/hyper-kvasir-labeled-images.zip'  # Change this!
zip_path = '/kaggle/input/chest-xray-pneumonia'  # Change this! /kaggle/input/chest-xray-pneumonia

if os.path.exists(zip_path):
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        zip_ref.extractall('/content/data')
    print("✓ Dataset extracted successfully!")

    # List contents
    !ls -R /content/data | head -20
else:
    print(f"❌ File not found: {zip_path}")
    print("Please upload your dataset or update the path")

IsADirectoryError: [Errno 21] Is a directory: '/kaggle/input/chest-xray-pneumonia'

### Verify Dataset Structure

In [ ]:
# Verify dataset structure
data_dir = '/content/data'  # Update this path if needed

def check_dataset_structure(data_dir):
    from pathlib import Path

    data_path = Path(data_dir)

    print("Checking dataset structure...\n")
    print("="*60)

    for split in ['train', 'val', 'test']:
        split_path = data_path / split
        if split_path.exists():
            pos_path = split_path / 'positive'
            neg_path = split_path / 'negative'

            n_pos = len(list(pos_path.glob('*'))) if pos_path.exists() else 0
            n_neg = len(list(neg_path.glob('*'))) if neg_path.exists() else 0
            total = n_pos + n_neg

            print(f"{split.upper()} SET:")
            print(f"  Total:    {total}")
            print(f"  Positive: {n_pos} ({100*n_pos/total:.1f}%)" if total > 0 else f"  Positive: {n_pos}")
            print(f"  Negative: {n_neg} ({100*n_neg/total:.1f}%)" if total > 0 else f"  Negative: {n_neg}")
            print()
        else:
            print(f"❌ {split.upper()} directory not found!\n")

    print("="*60)

check_dataset_structure(data_dir)

### Visualize Sample Images

In [ ]:
# Visualize sample images
import matplotlib.pyplot as plt
from PIL import Image
from pathlib import Path
import numpy as np

def visualize_samples(data_dir, split='train', num_samples=4):
    data_path = Path(data_dir) / split

    fig, axes = plt.subplots(2, num_samples, figsize=(3*num_samples, 6))

    # Show positive samples
    pos_dir = data_path / 'positive'
    if pos_dir.exists():
        pos_images = list(pos_dir.glob('*.jpg')) + list(pos_dir.glob('*.png'))
        for i, img_path in enumerate(pos_images[:num_samples]):
            img = Image.open(img_path).convert('RGB')
            axes[0, i].imshow(img)
            axes[0, i].set_title('H. pylori Positive', fontsize=10)
            axes[0, i].axis('off')

    # Show negative samples
    neg_dir = data_path / 'negative'
    if neg_dir.exists():
        neg_images = list(neg_dir.glob('*.jpg')) + list(neg_dir.glob('*.png'))
        for i, img_path in enumerate(neg_images[:num_samples]):
            img = Image.open(img_path).convert('RGB')
            axes[1, i].imshow(img)
            axes[1, i].set_title('H. pylori Negative', fontsize=10)
            axes[1, i].axis('off')

    plt.tight_layout()
    plt.savefig('/content/results/sample_images.png', dpi=100, bbox_inches='tight')
    plt.show()

visualize_samples(data_dir, split='train', num_samples=4)

## 3. Model Definitions

Define both baseline models with frozen backbones.

In [ ]:
# Model Definitions
import torch
import torch.nn as nn
from torchvision import models

class FrozenResNet50Baseline(nn.Module):
    """Baseline 1: Frozen ResNet-50 + Linear Classifier"""
    def __init__(self):
        super(FrozenResNet50Baseline, self).__init__()

        # Load pretrained ResNet-50
        resnet = models.resnet50(weights='IMAGENET1K_V1')

        # Freeze all parameters
        for param in resnet.parameters():
            param.requires_grad = False

        # Remove final FC layer
        self.backbone = nn.Sequential(*list(resnet.children())[:-1])

        # Add trainable linear classifier
        self.classifier = nn.Linear(2048, 1)

    def forward(self, x):
        with torch.no_grad():
            features = self.backbone(x)
        features = features.view(features.size(0), -1)
        logits = self.classifier(features)
        return logits


class FrozenDINOBaseline(nn.Module):
    """Baseline 2: Frozen DINO ViT + Linear Classifier"""
    def __init__(self, model_name='dinov2_vits14'):
        super(FrozenDINOBaseline, self).__init__()

        # Load DINO model from torch hub
        self.backbone = torch.hub.load('facebookresearch/dinov2', model_name)

        # Freeze all parameters
        for param in self.backbone.parameters():
            param.requires_grad = False

        # Get feature dimension
        feat_dim_map = {
            'dinov2_vits14': 384,
            'dinov2_vitb14': 768,
            'dinov2_vitl14': 1024,
            'dinov2_vitg14': 1536
        }
        feat_dim = feat_dim_map.get(model_name, 768)

        # Add trainable linear classifier
        self.classifier = nn.Linear(feat_dim, 1)

    def forward(self, x):
        with torch.no_grad():
            features = self.backbone(x)
        logits = self.classifier(features)
        return logits


print("✓ Model classes defined successfully!")

## 4. Data Loading

Create PyTorch datasets and dataloaders.

In [ ]:
# Data Loading
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
from pathlib import Path

class HPyloriDataset(Dataset):
    """Dataset for H. pylori detection"""
    def __init__(self, image_paths, labels, transform=None):
        self.image_paths = image_paths
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        image = Image.open(self.image_paths[idx]).convert('RGB')
        label = self.labels[idx]

        if self.transform:
            image = self.transform(image)

        return image, label


def load_data_from_directory(data_dir):
    """Load data from organized directory"""
    data_dir = Path(data_dir)
    splits = {}

    for split in ['train', 'val', 'test']:
        images = []
        labels = []

        # Load positive samples
        pos_dir = data_dir / split / 'positive'
        if pos_dir.exists():
            for img_path in pos_dir.glob('*'):
                if img_path.suffix.lower() in ['.jpg', '.jpeg', '.png']:
                    images.append(str(img_path))
                    labels.append(1)

        # Load negative samples
        neg_dir = data_dir / split / 'negative'
        if neg_dir.exists():
            for img_path in neg_dir.glob('*'):
                if img_path.suffix.lower() in ['.jpg', '.jpeg', '.png']:
                    images.append(str(img_path))
                    labels.append(0)

        splits[split] = {'images': images, 'labels': labels}

    return splits


def get_transforms(augment=False):
    """Get image transforms"""
    if augment:
        transform = transforms.Compose([
            transforms.Resize((256, 256)),
            transforms.RandomCrop((224, 224)),
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
            transforms.ToTensor(),
            transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
        ])
    else:
        transform = transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.ToTensor(),
            transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
        ])
    return transform


def create_dataloaders(data_dir, batch_size=32, num_workers=2):
    """Create PyTorch DataLoaders"""
    splits = load_data_from_directory(data_dir)

    train_transform = get_transforms(augment=True)
    test_transform = get_transforms(augment=False)

    train_dataset = HPyloriDataset(splits['train']['images'],
                                   splits['train']['labels'],
                                   train_transform)
    val_dataset = HPyloriDataset(splits['val']['images'],
                                 splits['val']['labels'],
                                 test_transform)
    test_dataset = HPyloriDataset(splits['test']['images'],
                                  splits['test']['labels'],
                                  test_transform)

    train_loader = DataLoader(train_dataset, batch_size=batch_size,
                             shuffle=True, num_workers=num_workers)
    val_loader = DataLoader(val_dataset, batch_size=batch_size,
                           shuffle=False, num_workers=num_workers)
    test_loader = DataLoader(test_dataset, batch_size=batch_size,
                            shuffle=False, num_workers=num_workers)

    print("="*60)
    print("DATASET LOADED")
    print("="*60)
    print(f"Training:   {len(train_dataset)} samples")
    print(f"Validation: {len(val_dataset)} samples")
    print(f"Test:       {len(test_dataset)} samples")
    print(f"Batch size: {batch_size}")
    print("="*60)

    return {'train': train_loader, 'val': val_loader, 'test': test_loader}


print("✓ Data loading utilities defined!")

## 5. Training Functions

Define training and evaluation functions.

In [ ]:
# Training and Evaluation Functions
import torch.optim as optim
from sklearn.metrics import roc_auc_score, accuracy_score, confusion_matrix
from sklearn.calibration import calibration_curve
from tqdm.notebook import tqdm
import numpy as np

def calculate_ece(y_true, y_probs, n_bins=10):
    """Calculate Expected Calibration Error"""
    bin_boundaries = np.linspace(0, 1, n_bins + 1)
    ece = 0.0

    for i in range(n_bins):
        bin_lower = bin_boundaries[i]
        bin_upper = bin_boundaries[i + 1]

        in_bin = (y_probs >= bin_lower) & (y_probs < bin_upper)
        bin_size = in_bin.sum()

        if bin_size > 0:
            bin_confidence = y_probs[in_bin].mean()
            bin_accuracy = y_true[in_bin].mean()
            ece += (bin_size / len(y_true)) * abs(bin_confidence - bin_accuracy)

    return ece


def train_epoch(model, dataloader, criterion, optimizer, device):
    """Train for one epoch"""
    model.train()
    total_loss = 0.0
    all_preds = []
    all_labels = []

    for images, labels in tqdm(dataloader, desc="Training", leave=False):
        images = images.to(device)
        labels = labels.float().to(device)

        optimizer.zero_grad()
        logits = model(images).squeeze()
        loss = criterion(logits, labels)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        probs = torch.sigmoid(logits)
        all_preds.extend(probs.detach().cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

    return total_loss / len(dataloader), np.array(all_preds), np.array(all_labels)


def evaluate(model, dataloader, criterion, device):
    """Evaluate model"""
    model.eval()
    total_loss = 0.0
    all_preds = []
    all_labels = []

    with torch.no_grad():
        for images, labels in tqdm(dataloader, desc="Evaluating", leave=False):
            images = images.to(device)
            labels = labels.float().to(device)

            logits = model(images).squeeze()
            loss = criterion(logits, labels)

            total_loss += loss.item()
            probs = torch.sigmoid(logits)
            all_preds.extend(probs.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    all_preds = np.array(all_preds)
    all_labels = np.array(all_labels)

    # Calculate metrics
    auc_roc = roc_auc_score(all_labels, all_preds)
    binary_preds = (all_preds >= 0.5).astype(int)
    accuracy = accuracy_score(all_labels, binary_preds)

    tn, fp, fn, tp = confusion_matrix(all_labels, binary_preds).ravel()
    sensitivity = tp / (tp + fn) if (tp + fn) > 0 else 0
    specificity = tn / (tn + fp) if (tn + fp) > 0 else 0
    ece = calculate_ece(all_labels, all_preds)

    metrics = {
        'loss': total_loss / len(dataloader),
        'auc_roc': auc_roc,
        'accuracy': accuracy,
        'sensitivity': sensitivity,
        'specificity': specificity,
        'ece': ece
    }

    return metrics, all_preds, all_labels


def train_baseline(model, train_loader, val_loader, test_loader,
                   model_name, epochs, lr, device):
    """Train a baseline model"""
    print(f"\n{'='*70}")
    print(f"Training {model_name}")
    print(f"{'='*70}\n")

    criterion = nn.BCEWithLogitsLoss()
    optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=lr)

    best_val_auc = 0.0
    best_epoch = 0
    history = {'train_loss': [], 'val_loss': [], 'val_auc': [], 'val_acc': []}

    for epoch in range(epochs):
        print(f"\nEpoch {epoch+1}/{epochs}")

        train_loss, _, _ = train_epoch(model, train_loader, criterion, optimizer, device)
        val_metrics, _, _ = evaluate(model, val_loader, criterion, device)

        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_metrics['loss'])
        history['val_auc'].append(val_metrics['auc_roc'])
        history['val_acc'].append(val_metrics['accuracy'])

        print(f"Train Loss: {train_loss:.4f}")
        print(f"Val   Loss: {val_metrics['loss']:.4f} | AUC: {val_metrics['auc_roc']:.4f} | "
              f"Acc: {val_metrics['accuracy']:.4f}")

        if val_metrics['auc_roc'] > best_val_auc:
            best_val_auc = val_metrics['auc_roc']
            best_epoch = epoch
            torch.save(model.state_dict(), f'/content/results/{model_name}_best.pth')
            print(f"✓ New best model saved (AUC: {best_val_auc:.4f})")

    # Load best model
    model.load_state_dict(torch.load(f'/content/results/{model_name}_best.pth'))

    # Test evaluation
    print(f"\n{'='*70}")
    print(f"Final Test Evaluation (Best model from epoch {best_epoch+1})")
    print(f"{'='*70}\n")

    test_metrics, test_preds, test_labels = evaluate(model, test_loader, criterion, device)

    print(f"Test Results:")
    print(f"  AUC-ROC:     {test_metrics['auc_roc']:.4f}")
    print(f"  Accuracy:    {test_metrics['accuracy']:.4f}")
    print(f"  Sensitivity: {test_metrics['sensitivity']:.4f}")
    print(f"  Specificity: {test_metrics['specificity']:.4f}")
    print(f"  ECE:         {test_metrics['ece']:.4f}")

    return {
        'history': history,
        'best_epoch': best_epoch,
        'test_metrics': test_metrics,
        'test_predictions': test_preds,
        'test_labels': test_labels
    }


print("✓ Training functions defined!")

## 6. Visualization Functions

In [ ]:
# Visualization Functions
import matplotlib.pyplot as plt
import json

def plot_results(results, save_dir='/content/results'):
    """Plot training curves and calibration"""
    for model_name, data in results.items():
        # Training curves
        fig, axes = plt.subplots(1, 2, figsize=(12, 4))

        # Loss
        axes[0].plot(data['history']['train_loss'], label='Train', linewidth=2)
        axes[0].plot(data['history']['val_loss'], label='Validation', linewidth=2)
        axes[0].set_xlabel('Epoch', fontsize=12)
        axes[0].set_ylabel('Loss', fontsize=12)
        axes[0].set_title(f'{model_name} - Loss', fontsize=14, fontweight='bold')
        axes[0].legend(fontsize=10)
        axes[0].grid(True, alpha=0.3)

        # AUC
        axes[1].plot(data['history']['val_auc'], label='Validation AUC', linewidth=2)
        axes[1].axhline(y=data['test_metrics']['auc_roc'], color='red',
                       linestyle='--', linewidth=2, label='Test AUC')
        axes[1].set_xlabel('Epoch', fontsize=12)
        axes[1].set_ylabel('AUC-ROC', fontsize=12)
        axes[1].set_title(f'{model_name} - AUC-ROC', fontsize=14, fontweight='bold')
        axes[1].legend(fontsize=10)
        axes[1].grid(True, alpha=0.3)

        plt.tight_layout()
        plt.savefig(f'{save_dir}/{model_name}_training.png', dpi=150)
        plt.show()

        # Calibration curve
        fig, ax = plt.subplots(figsize=(7, 7))
        fraction_of_positives, mean_predicted_value = calibration_curve(
            data['test_labels'], data['test_predictions'], n_bins=10
        )

        ax.plot(mean_predicted_value, fraction_of_positives, 's-',
                linewidth=2, markersize=8, label=model_name)
        ax.plot([0, 1], [0, 1], 'k--', linewidth=2, label='Perfect Calibration')
        ax.set_xlabel('Mean Predicted Probability', fontsize=12)
        ax.set_ylabel('Fraction of Positives', fontsize=12)
        ax.set_title(f'{model_name} - Calibration Curve\nECE: {data["test_metrics"]["ece"]:.4f}',
                    fontsize=14, fontweight='bold')
        ax.legend(fontsize=10)
        ax.grid(True, alpha=0.3)
        ax.set_xlim([0, 1])
        ax.set_ylim([0, 1])

        plt.tight_layout()
        plt.savefig(f'{save_dir}/{model_name}_calibration.png', dpi=150)
        plt.show()


def save_metrics(results, save_dir='/content/results'):
    """Save metrics to JSON"""
    metrics_summary = {}
    for model_name, data in results.items():
        metrics_summary[model_name] = {
            'test_auc_roc': float(data['test_metrics']['auc_roc']),
            'test_accuracy': float(data['test_metrics']['accuracy']),
            'test_sensitivity': float(data['test_metrics']['sensitivity']),
            'test_specificity': float(data['test_metrics']['specificity']),
            'test_ece': float(data['test_metrics']['ece']),
            'best_epoch': int(data['best_epoch'])
        }

    with open(f'{save_dir}/metrics_summary.json', 'w') as f:
        json.dump(metrics_summary, f, indent=2)

    print(f"\n✓ Metrics saved to {save_dir}/metrics_summary.json")


print("✓ Visualization functions defined!")

## 7. Configuration

Set up training configuration.

In [ ]:
# Configuration
CONFIG = {
    'data_dir': '/content/data',  # Update this path
    'batch_size': 32,
    'learning_rate': 1e-3,
    'epochs': 25,
    'num_workers': 2,
    'seed': 42,
    'output_dir': '/content/results'
}

# Set random seeds
torch.manual_seed(CONFIG['seed'])
np.random.seed(CONFIG['seed'])

# Device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print("="*70)
print("CONFIGURATION")
print("="*70)
print(f"Device:        {device}")
print(f"Data dir:      {CONFIG['data_dir']}")
print(f"Batch size:    {CONFIG['batch_size']}")
print(f"Learning rate: {CONFIG['learning_rate']}")
print(f"Epochs:        {CONFIG['epochs']}")
print(f"Output dir:    {CONFIG['output_dir']}")
print("="*70)

## 8. Load Data

Create the dataloaders.

In [ ]:
# Create dataloaders
dataloaders = create_dataloaders(
    CONFIG['data_dir'],
    batch_size=CONFIG['batch_size'],
    num_workers=CONFIG['num_workers']
)

## 9. Train Baseline 1: ResNet-50

Train the frozen ResNet-50 baseline.

In [ ]:
# Train ResNet-50 baseline
print("Loading ResNet-50 model...")
model_resnet = FrozenResNet50Baseline().to(device)

results_resnet = train_baseline(
    model_resnet,
    dataloaders['train'],
    dataloaders['val'],
    dataloaders['test'],
    'resnet50',
    CONFIG['epochs'],
    CONFIG['learning_rate'],
    device
)

## 10. Train Baseline 2: DINO ViT

Train the frozen DINO ViT baseline.

In [ ]:
# Train DINO baseline
print("Loading DINO ViT model...")
model_dino = FrozenDINOBaseline(model_name='dinov2_vits14').to(device)

results_dino = train_baseline(
    model_dino,
    dataloaders['train'],
    dataloaders['val'],
    dataloaders['test'],
    'dino',
    CONFIG['epochs'],
    CONFIG['learning_rate'],
    device
)

## 11. Visualize Results

Plot training curves and calibration.

In [ ]:
# Combine results
results = {
    'ResNet50': results_resnet,
    'DINO_ViT': results_dino
}

# Plot results
plot_results(results, CONFIG['output_dir'])

# Save metrics
save_metrics(results, CONFIG['output_dir'])

## 12. Final Comparison

Compare both baselines.

In [ ]:
# Print comparison
print("\n" + "="*70)
print("FINAL RESULTS COMPARISON")
print("="*70 + "\n")
print(f"{'Metric':<15} {'ResNet50':<12} {'DINO ViT':<12}")
print("-" * 40)
print(f"{'AUC-ROC':<15} {results_resnet['test_metrics']['auc_roc']:<12.4f} "
      f"{results_dino['test_metrics']['auc_roc']:<12.4f}")
print(f"{'Accuracy':<15} {results_resnet['test_metrics']['accuracy']:<12.4f} "
      f"{results_dino['test_metrics']['accuracy']:<12.4f}")
print(f"{'Sensitivity':<15} {results_resnet['test_metrics']['sensitivity']:<12.4f} "
      f"{results_dino['test_metrics']['sensitivity']:<12.4f}")
print(f"{'Specificity':<15} {results_resnet['test_metrics']['specificity']:<12.4f} "
      f"{results_dino['test_metrics']['specificity']:<12.4f}")
print(f"{'ECE':<15} {results_resnet['test_metrics']['ece']:<12.4f} "
      f"{results_dino['test_metrics']['ece']:<12.4f}")
print("\n" + "="*70)

# Determine best model
best_model = 'ResNet50' if results_resnet['test_metrics']['auc_roc'] > results_dino['test_metrics']['auc_roc'] else 'DINO ViT'
print(f"\n🏆 Best Model: {best_model}")
print("="*70)

## 13. Download Results

Download trained models and results.

In [ ]:
# List all result files
!ls -lh /content/results/

In [ ]:
# Create a ZIP file of all results
!cd /content && zip -r results.zip results/

print("\n✓ Results zipped! Download 'results.zip' from the file browser.")
print("  It contains:")
print("  - Trained model checkpoints (.pth)")
print("  - Training curves (.png)")
print("  - Calibration plots (.png)")
print("  - Metrics summary (JSON)")

In [ ]:
# Optional: Save to Google Drive
# Uncomment if you mounted Drive earlier

# !cp -r /content/results /content/drive/MyDrive/h_pylori_baselines/
# print("✓ Results copied to Google Drive!")

## Notes

### Key Points:
- Both models use frozen backbones (only linear layer trained)
- ResNet-50 is pre-trained on ImageNet (natural images)
- DINO ViT is self-supervised (often better for medical images)
- Training is fast (~2K parameters for ResNet, ~400 for DINO)

### Expected Performance:
- AUC-ROC: 0.75-0.88 (depends on dataset quality)
- Training time: 5-15 minutes per model (with GPU)

### Troubleshooting:
- **Out of memory**: Reduce `batch_size` in CONFIG
- **Slow training**: Ensure GPU is enabled (Runtime → Change runtime type → GPU)
- **Poor performance**: Check dataset labels and class balance

---

**Good luck with your H. pylori detection project! 🔬**